In [1]:
# sr_benchmark_run_ceql.py

from __future__ import annotations

import csv
import time
from pathlib import Path
from typing import Callable, Optional

import h5py
import numpy as np
import sympy as sp
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter

from config.benchmark_config import DataCFG, CEQL_TRAIN, CEQL
from src.ComplexEQL import ComplexEQL
from src.utils import set_seed, train_one_epoch


class RelativeMSELoss(nn.Module):
    def __init__(self, eps: float = 1e-6):
        super().__init__()
        self.eps = eps

    def forward(self, yhat: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
        r = yhat - y
        denom = y.abs() + self.eps
        return ((r / denom) ** 2).mean()


class MSEOrRelativeMSELoss(nn.Module):
    def __init__(self, eps: float = 1e-12, pivot: float = 1.0):
        super().__init__()
        self.eps = eps
        self.pivot = pivot

    def forward(self, yhat: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
        r = yhat - y
        denom = torch.maximum(y.abs(), y.new_tensor(self.pivot)) + self.eps
        return ((r / denom) ** 2).mean()


def _load_one_group(f: h5py.File, gname: str):
    g = f[gname]
    raw = g["sympy_str"][()]
    true_expr_str = raw.decode("utf-8") if isinstance(raw, (bytes, bytearray)) else str(raw)

    Xtr = g["train"]["X"][...].astype(np.float32, copy=False)
    ytr = g["train"]["y"][...].astype(np.float32, copy=False).reshape(-1)

    Xti = g["test_interp"]["X"][...].astype(np.float32, copy=False)
    yti = g["test_interp"]["y"][...].astype(np.float32, copy=False).reshape(-1)

    Xte = g["test_extrap"]["X"][...].astype(np.float32, copy=False)
    yte = g["test_extrap"]["y"][...].astype(np.float32, copy=False).reshape(-1)

    return true_expr_str, Xtr, ytr, Xti, yti, Xte, yte


def _mse(yhat: np.ndarray, y: np.ndarray) -> float:
    yhat = np.asarray(yhat, dtype=np.float64).reshape(-1)
    y = np.asarray(y, dtype=np.float64).reshape(-1)
    return float(np.mean((yhat - y) ** 2))


def _save_scatter_gif_x1x2(
    *,
    x1: np.ndarray,
    x2: np.ndarray,
    y_true: np.ndarray,
    y_pred_frames: list[np.ndarray],
    epoch_frames: list[int],
    gif_path: Path,
    title_prefix: str,
    fps: int = 2,
):
    x1 = np.asarray(x1, dtype=np.float64).reshape(-1)
    x2 = np.asarray(x2, dtype=np.float64).reshape(-1)
    y_true = np.asarray(y_true, dtype=np.float64).reshape(-1)

    y_preds_all = np.concatenate([np.asarray(p, dtype=np.float64).reshape(-1) for p in y_pred_frames], axis=0)

    x1_lo, x1_hi = float(x1.min()), float(x1.max())
    x2_lo, x2_hi = float(x2.min()), float(x2.max())
    y_lo = float(min(y_true.min(), y_preds_all.min()))
    y_hi = float(max(y_true.max(), y_preds_all.max()))

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4), constrained_layout=True)

    ax1.scatter(x1, y_true)
    sc1_pred = ax1.scatter(x1, y_pred_frames[0])
    ax1.set_xlabel("x1")
    ax1.set_ylabel("y")
    ax1.set_xlim(x1_lo, x1_hi)
    ax1.set_ylim(y_lo, y_hi)

    ax2.scatter(x2, y_true)
    sc2_pred = ax2.scatter(x2, y_pred_frames[0])
    ax2.set_xlabel("x2")
    ax2.set_ylabel("y")
    ax2.set_xlim(x2_lo, x2_hi)
    ax2.set_ylim(y_lo, y_hi)

    title = fig.suptitle(f"{title_prefix} | epoch={epoch_frames[0]}")

    def _update(i: int):
        yi = np.asarray(y_pred_frames[i], dtype=np.float64).reshape(-1)
        sc1_pred.set_offsets(np.c_[x1, yi])
        sc2_pred.set_offsets(np.c_[x2, yi])
        title.set_text(f"{title_prefix} | epoch={epoch_frames[i]}")
        return sc1_pred, sc2_pred, title

    anim = FuncAnimation(fig, _update, frames=len(y_pred_frames), interval=1000 // max(1, fps), blit=False)
    anim.save(gif_path, writer=PillowWriter(fps=fps))
    plt.close(fig)


def train_with_frames(
    *,
    model: torch.nn.Module,
    dataloader: DataLoader,
    optimizer: torch.optim.Optimizer,
    loss_fn: Callable,
    cfg,
    device: torch.device | str,
    scheduler=None,
    on_print: Optional[Callable[[int, torch.nn.Module], None]] = None,  # (epoch_1based, model)
):
    device = torch.device(device)
    global_epoch = 0

    data_losses: list[float] = []
    imag_w_losses: list[float] = []

    opt = optimizer
    sch = scheduler

    def _rebuild_optimizer_like(old_opt: torch.optim.Optimizer, new_model: torch.nn.Module) -> torch.optim.Optimizer:
        if isinstance(old_opt, torch.optim.Adam):
            return torch.optim.Adam(
                new_model.parameters(),
                lr=old_opt.param_groups[0]["lr"],
                betas=old_opt.param_groups[0].get("betas", (0.9, 0.999)),
                eps=old_opt.param_groups[0].get("eps", 1e-8),
                weight_decay=old_opt.param_groups[0].get("weight_decay", 0.0),
                amsgrad=old_opt.param_groups[0].get("amsgrad", False),
            )
        if isinstance(old_opt, torch.optim.AdamW):
            return torch.optim.AdamW(
                new_model.parameters(),
                lr=old_opt.param_groups[0]["lr"],
                betas=old_opt.param_groups[0].get("betas", (0.9, 0.999)),
                eps=old_opt.param_groups[0].get("eps", 1e-8),
                weight_decay=old_opt.param_groups[0].get("weight_decay", 0.01),
                amsgrad=old_opt.param_groups[0].get("amsgrad", False),
            )
        raise ValueError(f"Unsupported optimizer type for rebuild: {type(old_opt)}")

    def _rebuild_scheduler_like(old_sch, new_opt: torch.optim.Optimizer, cfg):
        if old_sch is None:
            return None
        if isinstance(old_sch, torch.optim.lr_scheduler.ReduceLROnPlateau):
            return torch.optim.lr_scheduler.ReduceLROnPlateau(new_opt, **getattr(cfg, "schedulerparams", {}))
        raise ValueError(f"Unsupported scheduler type for rebuild: {type(old_sch)}")

    def _record(avg_data: float, avg_imag_w_reg: float):
        data_losses.append(float(avg_data))
        imag_w_losses.append(float(avg_imag_w_reg))

    def _maybe_print(tag: str, avg_total: float, avg_data: float, avg_sparse: float, avg_imag_w: float):
        if (global_epoch + 1) % cfg.print_every == 0 or global_epoch == 0:
            lr = opt.param_groups[0]["lr"]
            active = model.count_active_edges()
            print(
                f"[{tag} | Epoch {global_epoch+1}] "
                f"lr={lr:.2e}, total={avg_total:.4e}, data={avg_data:.4e}, "
                f"sparsity_reg={avg_sparse:.4e}, imag_w={avg_imag_w:.4e}, "
                f"active_edges={active}"
            )
            if on_print is not None:
                on_print(int(global_epoch + 1), model)

    thr_min = float(getattr(cfg, "pruning_threshold_min", 0.0))
    thr_max = float(getattr(cfg, "pruning_threshold_max", float("inf")))
    min_edges_layer = int(getattr(cfg, "pruning_min_edges_per_layer", 0))

    def _do_prune_and_rebuild(tag: str, prune_fraction: float):
        nonlocal model, opt, sch

        before_total = model.count_active_edges()
        before_sym, before_asm = model.count_active_edges_per_layer()

        per_sym, asm = model.pruning_thresholds_from_fraction_per_layer(
            prune_fraction,
            min_edges_per_layer=min_edges_layer,
            eps=1e-12,
        )

        pruned_total = 0

        for li, (thr, k_to_prune, active_now) in enumerate(per_sym):
            if thr is None or k_to_prune <= 0 or active_now <= min_edges_layer:
                continue

            thr = float(thr)
            if thr < thr_min:
                thr = thr_min
            if thr > thr_max:
                thr = thr_max

            pruned_here = model.symbolic_layers[li].prune_by_threshold(thr)
            pruned_total += pruned_here
            if pruned_here > 0:
                after_li = int((model.symbolic_layers[li].mask > 0.5).sum().item())
                print(
                    f"[PRUNE_SYM | {tag} | layer={li}] "
                    f"pruned={pruned_here} (fraction={prune_fraction:g}, thr={thr:g}) "
                    f"active {before_sym[li]}->{after_li}"
                )

        thrA, kA, activeA = asm
        if thrA is not None and kA > 0 and activeA > min_edges_layer:
            thrA = float(thrA)
            if thrA < thr_min:
                thrA = thr_min
            if thrA > thr_max:
                thrA = thr_max

            prunedA = model.assembly_layer.prune_by_threshold(thrA)
            pruned_total += prunedA
            if prunedA > 0:
                afterA = int((model.assembly_layer.mask > 0.5).sum().item())
                print(
                    f"[PRUNE_ASM | {tag}] "
                    f"pruned={prunedA} (fraction={prune_fraction:g}, thr={thrA:g}) "
                    f"active {before_asm}->{afterA}"
                )

        dropped = model.drop_div_ops_with_pruned_denominator_()
        if dropped > 0:
            print(f"[DROP_DIV | {tag}] pruned_downstream_edges={dropped}")

        cleaned = model.cascade_cleanup_disconnected_()
        if cleaned > 0:
            print(f"[CLEAN | {tag}] cleaned_disconnected={cleaned}")

        before_rebuild_total = model.count_active_edges()
        model = model.rebuild_from_pruned().to(device)
        opt = _rebuild_optimizer_like(opt, model)
        sch = _rebuild_scheduler_like(sch, opt, cfg)

        after_total = model.count_active_edges()
        if pruned_total > 0:
            print(f"[PRUNE | {tag}] pruned_total={pruned_total} active {before_total}->{after_total}")
        else:
            if after_total != before_rebuild_total:
                print(f"[REBUILD | {tag}] active {before_rebuild_total}->{after_total}")

    phase1_epochs = int(getattr(cfg, "phase1_epochs", 0))
    for _ in range(phase1_epochs):
        avg_total, avg_data, _avg_reg, avg_sparse, avg_imag_w, _ = train_one_epoch(
            epoch=global_epoch,
            dataloader=dataloader,
            model=model,
            loss_fn=loss_fn,
            optimizer=opt,
            device=device,
            cfg=cfg,
            op_params_override=None,
            l1_enabled=True,
            l1_use_real_only=cfg.l1_on_real_only,
            l1_coeff=float(getattr(cfg, "l1_reg_coeff_phase1", 0.0)),
            l1_eps=float(getattr(cfg, "l1_eps", 1e-12)),
            imag_weights_penalty_enabled=True,
            imag_weights_penalty_coeff=float(getattr(cfg, "imag_w_coeff_phase1", 0.0)),
            prune_now=False,
            prune_threshold=0.0,
            normalize_divisions=False,
            normalize_divisions_eps=cfg.normalize_divisions_eps,
            clamp_pred=getattr(cfg, "clamp_pred", False),
            clamp_limit=getattr(cfg, "clamp_limit", 1e15),
            imag_shrink_enabled=False,
            imag_shrink_coeff=1.0,
        )
        _maybe_print("PHASE1", avg_total, avg_data, avg_sparse, avg_imag_w)
        _record(avg_data, avg_imag_w)
        global_epoch += 1

    phase1_prune_enabled = bool(getattr(cfg, "phase1_prune_enabled", True))
    phase1_prune_thr = float(getattr(cfg, "phase1_prune_threshold", 0.0))
    if phase1_prune_enabled and phase1_prune_thr > 0.0:
        tag = "PHASE1_END"
        model.normalize_all_divisions_(eps=float(getattr(cfg, "normalize_divisions_eps", 1e-12)))

        before = model.count_active_edges()
        pruned = model.prune_by_threshold(phase1_prune_thr)
        _ = model.drop_div_ops_with_pruned_denominator_()
        cleaned = model.cascade_cleanup_disconnected_()

        before_rebuild = model.count_active_edges()
        model = model.rebuild_from_pruned().to(device)
        opt = _rebuild_optimizer_like(opt, model)
        sch = _rebuild_scheduler_like(sch, opt, cfg)

        after = model.count_active_edges()
        model.normalize_all_divisions_(eps=float(getattr(cfg, "normalize_divisions_eps", 1e-12)))

        print(
            f"[{tag}] thr={phase1_prune_thr:g} pruned={pruned} cleaned={cleaned} "
            f"active {before}->{after} (pre_rebuild={before_rebuild})"
        )

    phase2_epochs = int(getattr(cfg, "phase2_epochs", 0))
    prune_every = int(getattr(cfg, "prune_every_epochs", 0))
    prune_fraction = float(getattr(cfg, "pruning_fraction_phase2", 0.0))
    normalize_divs = bool(getattr(cfg, "normalize_divisions_during_phase2", True))

    for e in range(phase2_epochs):
        avg_total, avg_data, _avg_reg, avg_sparse, avg_imag_w, _ = train_one_epoch(
            epoch=global_epoch,
            dataloader=dataloader,
            model=model,
            loss_fn=loss_fn,
            optimizer=opt,
            device=device,
            cfg=cfg,
            op_params_override=None,
            l1_enabled=True,
            l1_use_real_only=cfg.l1_on_real_only,
            l1_coeff=float(getattr(cfg, "l1_reg_coeff_phase2", 0.0)),
            l1_eps=float(getattr(cfg, "l1_eps", 1e-12)),
            imag_weights_penalty_enabled=True,
            imag_weights_penalty_coeff=float(getattr(cfg, "imag_w_coeff_phase2", 0.0)),
            prune_now=False,
            prune_threshold=0.0,
            normalize_divisions=normalize_divs,
            normalize_divisions_eps=cfg.normalize_divisions_eps,
            clamp_pred=getattr(cfg, "clamp_pred", True),
            clamp_limit=getattr(cfg, "clamp_limit", 1e15),
            imag_shrink_enabled=False,
            imag_shrink_coeff=1.0,
        )
        _maybe_print("PHASE2", avg_total, avg_data, avg_sparse, avg_imag_w)
        _record(avg_data, avg_imag_w)
        global_epoch += 1

        prune_warmup = int(getattr(cfg, "phase2_prune_warmup_epochs", 0))
        if (
            prune_every > 0
            and prune_fraction > 0.0
            and (e + 1) >= prune_warmup
            and ((e + 1 - prune_warmup) % prune_every == 0)
        ):
            _do_prune_and_rebuild(tag=f"PHASE2_E{e+1}", prune_fraction=prune_fraction)

    phase3_epochs = int(getattr(cfg, "phase3_epochs", 0))
    for _ in range(phase3_epochs):
        avg_total, avg_data, _avg_reg, avg_sparse, avg_imag_w, _ = train_one_epoch(
            epoch=global_epoch,
            dataloader=dataloader,
            model=model,
            loss_fn=loss_fn,
            optimizer=opt,
            device=device,
            cfg=cfg,
            op_params_override=None,
            l1_enabled=bool(getattr(cfg, "phase3_l1_enabled", False)),
            l1_use_real_only=cfg.l1_on_real_only,
            l1_coeff=float(getattr(cfg, "l1_reg_coeff_phase3", 0.0)),
            l1_eps=float(getattr(cfg, "l1_eps", 1e-12)),
            imag_weights_penalty_enabled=True,
            imag_weights_penalty_coeff=float(getattr(cfg, "imag_w_coeff_phase3", 0.0)),
            prune_now=False,
            prune_threshold=0.0,
            normalize_divisions=False,
            normalize_divisions_eps=cfg.normalize_divisions_eps,
            clamp_pred=getattr(cfg, "clamp_pred", True),
            clamp_limit=getattr(cfg, "clamp_limit", 1e15),
            imag_shrink_enabled=cfg.phase3_imag_shrink_enabled,
            imag_shrink_coeff=cfg.phase3_imag_shrink_coeff,
        )

        if sch is not None:
            try:
                sch.step(avg_data)
            except TypeError:
                sch.step()

        _maybe_print("PHASE3", avg_total, avg_data, avg_sparse, avg_imag_w)
        _record(avg_data, avg_imag_w)
        global_epoch += 1

    return model, (imag_w_losses, data_losses)


def main():
    cfg = DataCFG()

    out_csv = Path(getattr(CEQL_TRAIN, "results_path", "reports/sr_benchmark_ceql.csv"))
    out_csv.parent.mkdir(parents=True, exist_ok=True)

    n_runs = int(getattr(CEQL_TRAIN, "n_runs", 1))
    base_seed = int(getattr(CEQL_TRAIN, "base_seed", 0))

    device = torch.device(getattr(CEQL_TRAIN, "device", "cpu"))

    script_dir = Path.cwd()

    with h5py.File(cfg.h5_path, "r") as f, out_csv.open("w", newline="") as out:
        w = csv.writer(out)
        w.writerow(
            [
                "group",
                "run",
                "seed",
                "n_train",
                "n_features",
                "train_mse",
                "test_interp_mse",
                "test_extrap_mse",
                "duration_s",
                "true_expr",
                "found_expr",
            ]
        )

        groups = sorted(list(f.keys()))
        print(groups)
        for gname in groups:
            if gname in [
                "expr_000_lin_uni",
                "expr_001_lin_bi",
                "expr_002_poly2_uni",
                "expr_003_poly2_bi",
                "expr_004_lin_pm_log",
                "expr_005_lin_pm_sqrt",
                "expr_006_lin_plus_pow",
                "expr_007_rat_linlin_pole_extrap_only",
                "expr_008_rat_linlin_pole_train_only",
                "expr_009_rat_poly2poly2_pole_extrap_only",
            ]:
                continue

            true_expr_str, Xtr, ytr, Xti, yti, Xte, yte = _load_one_group(f, gname)

            n_features = int(Xtr.shape[1])
            CEQL.n_input_fields = n_features

            Xtr_t = torch.tensor(Xtr, device=device)
            ytr_t = torch.tensor(ytr.reshape(-1, 1), device=device)

            dl = DataLoader(
                TensorDataset(Xtr_t, ytr_t),
                batch_size=int(getattr(CEQL_TRAIN, "train_batch_size", 2**14)),
                shuffle=True,
                drop_last=False,
            )

            for run_i in range(n_runs):
                seed = base_seed + run_i
                set_seed(seed)

                model = ComplexEQL(CEQL).to(device)
                loss_fn = MSEOrRelativeMSELoss(eps=1e-12, pivot=1.0)

                opt = torch.optim.Adam(model.parameters(), lr=float(getattr(CEQL_TRAIN, "lr", 1e-3)))

                sched = None
                if getattr(CEQL_TRAIN, "scheduler", None) == "ReduceLROnPlateau":
                    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(
                        opt, **getattr(CEQL_TRAIN, "schedulerparams", {})
                    )

                yhat_frames: list[np.ndarray] = []
                epoch_frames: list[int] = []

                Xtr_pred_t = torch.tensor(Xtr, device=device)

                def on_print(epoch_1based: int, m: torch.nn.Module):
                    m.eval()
                    with torch.no_grad():
                        yhat = m(Xtr_pred_t).real.squeeze(-1).detach().cpu().numpy().copy()
                    yhat_frames.append(yhat)
                    epoch_frames.append(int(epoch_1based))
                    m.train()

                t0 = time.perf_counter()
                model, _ = train_with_frames(
                    model=model,
                    dataloader=dl,
                    optimizer=opt,
                    loss_fn=loss_fn,
                    cfg=CEQL_TRAIN,
                    device=device,
                    scheduler=sched,
                    on_print=on_print,
                )
                dur = time.perf_counter() - t0

                model.eval()
                with torch.no_grad():
                    yhat_tr = model(torch.tensor(Xtr, device=device)).real.squeeze(-1).cpu().numpy()
                    yhat_ti = model(torch.tensor(Xti, device=device)).real.squeeze(-1).cpu().numpy()
                    yhat_te = model(torch.tensor(Xte, device=device)).real.squeeze(-1).cpu().numpy()

                train_mse = _mse(yhat_tr, ytr)
                test_interp_mse = _mse(yhat_ti, yti)
                test_extrap_mse = _mse(yhat_te, yte)

                found_expr_str = ""
                try:
                    if n_features == 1:
                        syms = [sp.Symbol("x1")]
                    else:
                        syms = [sp.Symbol(f"x{i+1}") for i in range(n_features)]
                    found = model.get_symbolic_expression(syms, rounding_decimals=5, use_imag=True)
                    found_expr_str = "" if found is None else str(found)
                except Exception:
                    found_expr_str = ""

                w.writerow(
                    [
                        gname,
                        run_i,
                        seed,
                        int(Xtr.shape[0]),
                        n_features,
                        train_mse,
                        test_interp_mse,
                        test_extrap_mse,
                        dur,
                        true_expr_str,
                        found_expr_str,
                    ]
                )
                out.flush()

                print(
                    f"[{gname}] run={run_i} seed={seed} "
                    f"train={train_mse:.3e} interp={test_interp_mse:.3e} extrap={test_extrap_mse:.3e} "
                    f"dur={dur:.2f}s"
                )
                if found_expr_str:
                    print("Found:", found_expr_str)

                if len(yhat_frames) >= 2 and Xtr.shape[1] >= 2:
                    gif_path = script_dir / f"{gname}_run{run_i}_seed{seed}.gif"

                    x1 = Xtr[:, 0].reshape(-1)
                    x2 = Xtr[:, 1].reshape(-1)

                    _save_scatter_gif_x1x2(
                        x1=x1,
                        x2=x2,
                        y_true=ytr,
                        y_pred_frames=yhat_frames,
                        epoch_frames=epoch_frames,
                        gif_path=gif_path,
                        title_prefix=f"{gname} run={run_i} seed={seed}",
                        fps=2,
                    )
                    print(f"Saved GIF: {gif_path}")

    print(f"\nSaved: {out_csv}")


if __name__ == "__main__":
    main()


['expr_000_lin_uni', 'expr_001_lin_bi', 'expr_002_poly2_uni', 'expr_003_poly2_bi', 'expr_004_lin_pm_log', 'expr_005_lin_pm_sqrt', 'expr_006_lin_plus_pow', 'expr_007_rat_linlin_pole_extrap_only', 'expr_008_rat_linlin_pole_train_only', 'expr_009_rat_poly2poly2_pole_extrap_only', 'expr_010_rat_poly2poly2_pole_train_only']
Random seed set as 0
[PHASE1 | Epoch 1] lr=1.00e-03, total=7.4580e-01, data=7.4580e-01, sparsity_reg=3.2160e-09, imag_w=6.8802e-10, active_edges=93
[PHASE1 | Epoch 1000] lr=1.00e-03, total=2.5081e-01, data=2.5081e-01, sparsity_reg=3.9398e-09, imag_w=7.3201e-10, active_edges=93
[PHASE1 | Epoch 2000] lr=1.00e-03, total=5.9698e-03, data=5.9698e-03, sparsity_reg=3.8241e-09, imag_w=6.3321e-10, active_edges=93
[PHASE1 | Epoch 3000] lr=1.00e-03, total=3.7774e-04, data=3.7773e-04, sparsity_reg=3.8082e-09, imag_w=6.0630e-10, active_edges=93
[PHASE1 | Epoch 4000] lr=1.00e-03, total=4.1930e-05, data=4.1926e-05, sparsity_reg=3.7135e-09, imag_w=5.8713e-10, active_edges=93
[PHASE1 | E